#**Deep Natural Language Processing @ PoliTO**

---


**Teaching Assistants:** Giuseppe Gallipoli (part 1) and Ali Yassine (part 2)

**Credits:** Moreno La Quatra

**Practice 1:** Text processing (part 1) and Topic modeling (part 2)

# **Part 1: Text processing**
---
The text processing phase is a preliminary stage where the text to be manipulated is processed to be ready for subsequent analysis.

Text processing usually entails several steps that could possibly include:
- **Language Identification**: identifying the language of a given text.
- **Tokenization**: splitting a given text in several sentences/words.
- **Dependency tree parsing:** analyzing the depencies between words composing the text.
- **Stemming/Lemmatization:** obtain the root form for each word in text.
- **Stopword removal**: removing words that are si commonly used that they carry very little useful information.
- **Part of Speech Tagging:** given a word, retrieve its part of speech (proper noun, common noun or verb).



### Language Identification

| Text                                                                                                                                | Language Code |
|-------------------------------------------------------------------------------------------------------------------------------------|---------------|
| The "Deep Natural Language Processing" course is offered during the first semester of the second year at Politecnico di Torino      | `EN`            |
| Il corso "Deep Natural Language Processing" viene impartito al Politecnico di Torino durante il primo semestre del secondo anno.    | `IT`            |
| Le cours "Deep Natural Language Processing" est enseigné au Politecnico di Torino pendant le premier semestre de la deuxième année. | `FR`            |

**Language Identification** is a crucial prelimiary step because each language has its own characteristics. The knowledge of the main language associated to a given text could be beneficial for all subsequent steps in text processing pipeline.

The data collection used in this first part of the practice is provided [here](https://github.com/MorenoLaQuatra/DeepNLP/blob/main/practices/P1/langid_dataset.csv) - [source: Kaggle](https://www.kaggle.com/martinkk5575/language-detection)


# Exercise 1:

Benchmark different language-detection algorithm by computing the accuracy of each approach:
- [LangID](https://github.com/saffsd/langid.py)
- [langdetect](https://pypi.org/project/langdetect/)
- [fastlangid](https://pypi.org/project/fastlangid/) (built on FastText)

**Hint:** language code conversion: [iso639-lang](https://pypi.org/project/iso639-lang/)

For each method report:
- Accuracy
- Average time per example

In [22]:
#!wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P1/langid_dataset.csv

In [23]:
# your code here
!pip install langid langdetect fastlangid iso639-lang pandas scikit-learn

In [24]:
# your code here
import pandas as pd
import time
from sklearn.metrics import accuracy_score
import langid
from langdetect import detect as langdetect_detect
import fastlangid
# Using iso639-lang for language code conversion
from iso639 import Lang

# Load the dataset
df = pd.read_csv('langid_dataset.csv')

# Prepare data
texts = df['Text'].tolist()
true_labels = df['language'].tolist() # Corrected column name

def get_iso639_1(lang_name):
    try:
        lang_obj = Lang(lang_name)
        if lang_obj:
            return lang_obj.pt1
        return None
    except AttributeError:
        # Gestisce il caso in cui lang_obj non abbia l'attributo part1
        return None
    except Exception as e:
        # Log dell'errore per debug, se necessario
        print(f"Errore in get_iso639_1: {e}")
        return None

# Convert true labels to ISO 639-1 for comparison
true_labels_iso639_1 = [get_iso639_1(lang) for lang in true_labels]

# Remove entries where ISO 639-1 conversion failed (if any)
valid_indices = [i for i, label in enumerate(true_labels_iso639_1) if label is not None]
texts_valid = [texts[i] for i in valid_indices]
true_labels_valid = [true_labels_iso639_1[i] for i in valid_indices]

# Benchmark LangID
langid_predictions = []
start_time = time.time()
for text in texts_valid:
    try:
        lang, _ = langid.classify(text)
        langid_predictions.append(lang)
    except:
        langid_predictions.append(None) # Handle potential errors

end_time = time.time()

if len(texts_valid) > 0:
    avg_time_langid = (end_time - start_time) / len(texts_valid)
    accuracy_langid = accuracy_score(true_labels_valid, langid_predictions)

    print(f"LangID:")
    print(f"  Accuracy: {accuracy_langid:.4f}")
    print(f"  Average time per example: {avg_time_langid:.6f} seconds")
else:
    print("LangID: No valid texts found for benchmarking.")

# Benchmark langdetect
langdetect_predictions = []
start_time = time.time()
for text in texts_valid:
    try:
        langdetect_predictions.append(langdetect_detect(text))
    except:
        langdetect_predictions.append(None) # Handle potential errors

end_time = time.time()

if len(texts_valid) > 0:
    avg_time_langdetect = (end_time - start_time) / len(texts_valid)
    # Align predictions with valid true labels, handling None values
    aligned_langdetect_predictions = []
    aligned_true_labels = []
    for i, pred in enumerate(langdetect_predictions):
        if pred is not None:
            aligned_langdetect_predictions.append(pred)
            aligned_true_labels.append(true_labels_valid[i]) # Assuming the original index corresponds

    accuracy_langdetect = accuracy_score(aligned_true_labels, aligned_langdetect_predictions)


    print(f"\nlangdetect:")
    print(f"  Accuracy: {accuracy_langdetect:.4f}")
    print(f"  Average time per example: {avg_time_langdetect:.6f} seconds")
else:
    print("\nlangdetect: No valid texts found for benchmarking.")

# Benchmark fastlangid
fastlid = fastlangid.langid.LID()
fastlangid_predictions = []
start_time = time.time()
for text in texts_valid:
    try:
        lang, _ = fastlid.predict(text)
        fastlangid_predictions.append(lang)
    except:
        fastlangid_predictions.append(None) # Handle potential errors

end_time = time.time()
if len(texts_valid) > 0:
    avg_time_fastlangid = (end_time - start_time) / len(texts_valid)
    accuracy_fastlangid = accuracy_score(true_labels_valid, fastlangid_predictions)

    print(f"\nfastlangid:")
    print(f"  Accuracy: {accuracy_fastlangid:.4f}")
    print(f"  Average time per example: {avg_time_fastlangid:.6f} seconds")
else:
    print("\nfastlangid: No valid texts found for benchmarking.")

LangID:
  Accuracy: 0.9543
  Average time per example: 0.002255 seconds

langdetect:
  Accuracy: 0.8436
  Average time per example: 0.001424 seconds


ValueError: Classification metrics can't handle a mix of multiclass and unknown targets

In [ ]:
# Note: To ensure compatibility with fastlangid, you may need to uninstall the current numpy version and downgrade to an earlier one (e.g., numpy < 2.0).
# After installation, please restart the session to activate the newly installed numpy version.

# Exercise 2

For English-written text, apply word-level tokenization. What is the average number of words per sentence?

Implement word-tokenization using both [nltk](https://www.nltk.org/) and [spacy](https://spacy.io/). Report the results for both of them.

For spaCy use the `en_core_web_sm` model.

In [ ]:
# your code here
!pip install nltk spacy

In [25]:
# your code here
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
import spacy
import time # Import the time module

# Download the necessary resources
nltk.download('punkt_tab')

# Load the spaCy English model
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Downloading spaCy model 'en_core_web_sm'...")
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

# Filter for English texts
english_texts = [text for text, lang in zip(df['Text'], true_labels_valid) if lang == 'en']

print(f"Number of English texts: {len(english_texts)}")

# NLTK Tokenization
nltk_word_counts = []
nltk_sentence_counts = []
start_time_nltk = time.time()
for text in english_texts:
    sentences = sent_tokenize(text)
    nltk_sentence_counts.append(len(sentences))
    for sentence in sentences:
        words = word_tokenize(sentence)
        nltk_word_counts.append(len(words))
end_time_nltk = time.time()

avg_time_nltk = (end_time_nltk - start_time_nltk) / len(english_texts) if len(english_texts) > 0 else 0
avg_words_per_sentence_nltk = sum(nltk_word_counts) / sum(nltk_sentence_counts) if sum(nltk_sentence_counts) > 0 else 0

print("\nNLTK Tokenization:")
print(f"  Average time per example: {avg_time_nltk:.6f} seconds")
print(f"  Average words per sentence: {avg_words_per_sentence_nltk:.2f}")

# spaCy Tokenization
spacy_word_counts = []
spacy_sentence_counts = []
start_time_spacy = time.time()
for text in english_texts:
    doc = nlp(text)
    sentences = list(doc.sents)
    spacy_sentence_counts.append(len(sentences))
    for sentence in sentences:
        spacy_word_counts.append(len([token for token in sentence if not token.is_punct])) # Exclude punctuation from word count

end_time_spacy = time.time()

avg_time_spacy = (end_time_spacy - start_time_spacy) / len(english_texts) if len(english_texts) > 0 else 0
avg_words_per_sentence_spacy = sum(spacy_word_counts) / sum(spacy_sentence_counts) if sum(spacy_sentence_counts) > 0 else 0

print("\nspaCy Tokenization:")
print(f"  Average time per example: {avg_time_spacy:.6f} seconds")
print(f"  Average words per sentence: {avg_words_per_sentence_spacy:.2f}")

[nltk_data] Downloading package punkt_tab to /home/marc-antonio-
[nltk_data]     lopez/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Number of English texts: 1000

NLTK Tokenization:
  Average time per example: 0.000117 seconds
  Average words per sentence: 68.27

spaCy Tokenization:
  Average time per example: 0.010765 seconds
  Average words per sentence: 65.20


# Exercise 3

Dependency Parsing aims at analyzing the grammatical structure of sentences. The main goal is to find out related words as well as the type of the relationship between them.

The output of this step is a dependency tree similar to the one reported in the figure below.

![dependency tree](http://www.rangakrish.com/wp-content/uploads/2018/04/Deptree-example2.png)

Use spacy to parse the dependency tree of a **randomly selected** sentence. You can both use English sentences or your native language (if supported in [spaCy](https://spacy.io/usage/models/)). Use [displaCy](https://explosion.ai/demos/displacy) to visualize the result in the notebook.

In [26]:
# your code here
import random
from spacy import displacy
from IPython.display import display, HTML

# Select a random English sentence
if english_texts:
    random_sentence = random.choice(english_texts)
    print(f"Randomly selected sentence: {random_sentence}")

    # Process the sentence with spaCy
    doc = nlp(random_sentence)

    # Generate HTML and display manually
    html = displacy.render(doc, style="dep", jupyter=False)
    display(HTML(html))
else:
    print("No English texts available to select a random sentence.")

Randomly selected sentence: demotic self-reliance is meant in terms of radical decentralisation and collective self-sufficiency in the sense of relying on the demos resources rather than in the sense of autarky


# Exercise 4
For the same sentence selected in the previous step apply all the following steps:
1. Lemmatization: convert each word to its root form.
2. Stopword removal: remove language-specific stopwords.
3. Part of Speech Tagging: for each word in the sentence display its part-of-speech.

For each step, print the resulting list on the console.

In [27]:
# your code here
# Make sure 'random_sentence' and 'nlp' are defined from previous cells
if 'random_sentence' in locals() and 'nlp' in locals():
    doc = nlp(random_sentence)

    # 1. Lemmatization
    lemmatized_tokens = [token.lemma_ for token in doc]
    print("Lemmatized tokens:")
    print(lemmatized_tokens)

    # 2. Stopword removal
    filtered_tokens = [token.text for token in doc if not token.is_stop]
    print("\nTokens after stopword removal:")
    print(filtered_tokens)

    # 3. Part of Speech Tagging
    pos_tags = [(token.text, token.pos_) for token in doc]
    print("\nPart of Speech Tags:")
    print(pos_tags)
else:
    print("Please run the previous cells to define 'random_sentence' and 'nlp'.")

Lemmatized tokens:
['demotic', 'self', '-', 'reliance', 'be', 'mean', 'in', 'term', 'of', 'radical', 'decentralisation', 'and', 'collective', 'self', '-', 'sufficiency', 'in', 'the', 'sense', 'of', 'rely', 'on', 'the', 'demos', 'resource', 'rather', 'than', 'in', 'the', 'sense', 'of', 'autarky']

Tokens after stopword removal:
['demotic', 'self', '-', 'reliance', 'meant', 'terms', 'radical', 'decentralisation', 'collective', 'self', '-', 'sufficiency', 'sense', 'relying', 'demos', 'resources', 'sense', 'autarky']

Part of Speech Tags:
[('demotic', 'ADJ'), ('self', 'NOUN'), ('-', 'PUNCT'), ('reliance', 'NOUN'), ('is', 'AUX'), ('meant', 'VERB'), ('in', 'ADP'), ('terms', 'NOUN'), ('of', 'ADP'), ('radical', 'ADJ'), ('decentralisation', 'NOUN'), ('and', 'CCONJ'), ('collective', 'ADJ'), ('self', 'NOUN'), ('-', 'PUNCT'), ('sufficiency', 'NOUN'), ('in', 'ADP'), ('the', 'DET'), ('sense', 'NOUN'), ('of', 'ADP'), ('relying', 'VERB'), ('on', 'ADP'), ('the', 'DET'), ('demos', 'PROPN'), ('resources'

# **Occurrence-based text representation - TF-IDF**

---

TF-IDF (term frequency-inverse document frequency) is a statistical measure that evaluates how relevant a word is to a document in a collection of documents. It allows to create occurrence-based vector representation for each document.

# Exercise 5
Use TF-IDF to vectorize each sentence in the original data collection. You can choose your preferred implementation for TF-IDF vectorization. It is also available on [SciKit-Learn library](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [29]:
# your code here
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Build a validated dataframe aligned with texts_valid / true_labels_valid created earlier
df_valid = pd.DataFrame({"Text": texts_valid, "lang": true_labels_valid})

# Keep only English texts
df_english = df_valid[df_valid["lang"] == "en"].reset_index(drop=True)
print(f"English dataset shape: {df_english.shape}")
print(f"Number of English documents: {len(df_english)}")

if len(df_english) == 0:
    print("No English texts available for TF-IDF.")
else:
    # Initialize TF-IDF Vectorizer (English texts only)
    tfidf_vectorizer = TfidfVectorizer(
        max_features=5000,
        min_df=2,
        max_df=0.8,
        lowercase=True,
        stop_words="english"
    )

    # Fit and transform the English text data
    print("Fitting TF-IDF vectorizer on the English text data...")
    tfidf_matrix = tfidf_vectorizer.fit_transform(df_english["Text"])

    print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
    print(f"Number of documents (sentences): {tfidf_matrix.shape[0]}")
    print(f"Number of features (vocabulary size): {tfidf_matrix.shape[1]}")

    # Get feature names (vocabulary)
    feature_names = tfidf_vectorizer.get_feature_names_out()
    print(f"\nFirst 20 features in vocabulary: {list(feature_names[:20])}")

    # Show TF-IDF values for a few sample documents
    print(f"\nTF-IDF representation for first 3 documents:")
    for i in range(min(3, len(df_english))):
        doc_vector = tfidf_matrix[i]
        feature_indices = doc_vector.nonzero()[1]
        tfidf_scores = [(feature_names[idx], doc_vector[0, idx]) for idx in feature_indices]
        tfidf_scores.sort(key=lambda x: x[1], reverse=True)
        print(f"\nDocument {i+1}: {df_english['Text'].iloc[i][:100]}...")
        print(f"Top 5 terms by TF-IDF: {tfidf_scores[:5]}")

    # Show some statistics about the TF-IDF matrix
    print(f"\nTF-IDF Matrix Statistics:")
    print(f"Non-zero elements: {tfidf_matrix.nnz}")
    print(f"Sparsity: {1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.4f}")
    print(f"Average TF-IDF value: {tfidf_matrix.mean():.6f}")
    print(f"Max TF-IDF value: {tfidf_matrix.max():.6f}")

English dataset shape: (1000, 2)
Number of English documents: 1000
Fitting TF-IDF vectorizer on the English text data...
TF-IDF matrix shape: (1000, 4705)
Number of documents (sentences): 1000
Number of features (vocabulary size): 4705

First 20 features in vocabulary: ['abandoned', 'abd', 'abducted', 'abduction', 'abdul', 'abdullah', 'aberllefenni', 'abilities', 'able', 'aboard', 'abraham', 'abruptly', 'absorbed', 'abstract', 'abundance', 'academic', 'academy', 'accept', 'accepted', 'access']

TF-IDF representation for first 3 documents:

Document 1: in  johnson was awarded an american institute of architects gold medal in  he became the first recip...
Top 5 terms by TF-IDF: [('recipient', np.float64(0.3101259381468956)), ('pritzker', np.float64(0.3101259381468956)), ('prestigious', np.float64(0.3101259381468956)), ('medal', np.float64(0.297025232437774)), ('architects', np.float64(0.2785608247971744))]

Document 2: bussy-saint-georges has built its identity on a green model environme

# Exercise 6

Build a supervised multi-class language detector using as features the vector obtained by TF-IDF representation. Use 80% of the data to train the language detector and 20% of the data for assessing its accuracy.

In [31]:
# your code here
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np

# Use the full dataset with all languages (from Exercise 1)
print(f"Total number of texts: {len(texts_valid)}")
print(f"Language distribution: {pd.Series(true_labels_valid).value_counts()}")

# Initialize TF-IDF Vectorizer for all languages
tfidf_vectorizer_all = TfidfVectorizer(
    max_features=3000,
    min_df=2,
    max_df=0.8,
    lowercase=True,
    ngram_range=(1, 2),  # Use unigrams and bigrams
    analyzer='char_wb'   # Character n-grams work better for language detection
)

# Fit and transform all texts
print("\nVectorizing all texts with TF-IDF...")
X = tfidf_vectorizer_all.fit_transform(texts_valid)
y = true_labels_valid

print(f"TF-IDF matrix shape: {X.shape}")
print(f"Number of unique languages: {len(set(y))}")

# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Ensure balanced splits
)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

# Train Logistic Regression classifier
print("\nTraining Logistic Regression classifier...")
classifier = LogisticRegression(
    max_iter=1000,
    random_state=42,
    solver='lbfgs',
    multi_class='multinomial'
)
classifier.fit(X_train, y_train)

# Make predictions on test set
y_pred = classifier.predict(X_test)

# Evaluate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\n{'='*60}")
print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"{'='*60}")

# Display detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# Display confusion matrix
print("\nConfusion Matrix:")
conf_matrix = confusion_matrix(y_test, y_pred)
languages = sorted(set(y_test))
conf_df = pd.DataFrame(conf_matrix, index=languages, columns=languages)
print(conf_df)

# Show some example predictions
print("\nSample Predictions:")
sample_indices = np.random.choice(X_test.shape[0], min(5, X_test.shape[0]), replace=False)
for idx in sample_indices:
    print(f"\nTrue: {y_test[idx]} | Predicted: {y_pred[idx]}")
    # Note: idx here refers to position in test set, not original dataset
    # To show text, we need to map back to original indices

Total number of texts: 22000
Language distribution: et    1000
sv    1000
th    1000
ta    1000
nl    1000
ja    1000
tr    1000
la    1000
ur    1000
id    1000
pt    1000
fr    1000
zh    1000
ko    1000
hi    1000
es    1000
ps    1000
fa    1000
ro    1000
ru    1000
en    1000
ar    1000
Name: count, dtype: int64

Vectorizing all texts with TF-IDF...
TF-IDF matrix shape: (22000, 3000)
Number of unique languages: 22

Training set size: 17600
Test set size: 4400

Training Logistic Regression classifier...


/home/marc-antonio-lopez/Documenti/github/polito/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



Test Accuracy: 0.9782 (97.82%)

Classification Report:
              precision    recall  f1-score   support

          ar       1.00      1.00      1.00       200
          en       0.77      0.98      0.87       200
          es       0.97      0.98      0.98       200
          et       0.99      0.96      0.98       200
          fa       0.99      1.00      1.00       200
          fr       0.97      0.99      0.98       200
          hi       1.00      0.96      0.98       200
          id       0.99      0.97      0.98       200
          ja       1.00      0.99      0.99       200
          ko       0.99      0.99      0.99       200
          la       0.96      0.94      0.95       200
          nl       0.98      0.99      0.99       200
          ps       1.00      0.91      0.95       200
          pt       0.97      0.94      0.96       200
          ro       1.00      0.98      0.99       200
          ru       0.99      0.99      0.99       200
          sv       0.99  